# env preparation

In [ ]:
# adapt to colab gpu / nvidia 5060laptop setup

import os
import sys
import torch
import warnings

# SAVED_MODEL_NAME = "baseline_model_focal"
# LOADED_MODEL_NAME = "baseline_model_focal"
SAVED_MODEL_NAME = "baseline_model_crf"
LOADED_MODEL_NAME = "baseline_model_crf"
MODEL_NAME = "DeepPavlov/rubert-base-cased-sentence"
MAX_LEN = 512
DEBUG = False

try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    print("We are in Colab!")

    # Mount Drive
    from google.colab import drive
    drive.mount('/content/drive')

    !pip install -q transformers datasets seqeval evaluate accelerate torchcrf -q

    ROOT_DIR = "/content/drive/MyDrive/omg_diploma_2025/restore_punct"
    DATA_DIR = os.path.join(ROOT_DIR, "data")
    MODEL_DIR = os.path.join(ROOT_DIR, "models")
    # MODEL_DIR = DATA_DIR
    BATCH_SIZE = 8
    NUM_WORKERS = 2
    GRAD_ACCUM = 2

    if ROOT_DIR not in sys.path:
        sys.path.append(ROOT_DIR)

else:
    print("We are running locally!")
    ROOT_DIR = os.getcwd()
    DATA_DIR = os.path.join(ROOT_DIR, "data")
    MODEL_DIR = os.path.join(ROOT_DIR, "models")
    # MODEL_DIR = DATA_DIR
    # RTX 5060 (8GB VRAM)
    BATCH_SIZE = 4
    NUM_WORKERS = 4
    GRAD_ACCUM = 4

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)


We are running locally!


In [ ]:
import json
import evaluate
import numpy as np
from transformers import AutoTokenizer, AutoConfig, AutoModel, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
from datasets import Dataset, load_dataset
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import KFold
from torchcrf import CRF
import torch.nn as nn
import pandas as pd


In [3]:
print(torch.cuda.is_available())
# !nvidia-smi


True


In [ ]:
warnings.filterwarnings("ignore", category=FutureWarning, message=".*`tokenizer` is deprecated.*")
warnings.filterwarnings("ignore", message=".*seems not to be NE tag.*")
warnings.filterwarnings("ignore", message=".*UndefinedMetricWarning.*")


In [ ]:
data_files = {
    "train": os.path.join(DATA_DIR, "train_all.json"),
    "validation": os.path.join(DATA_DIR, "val_internal.json")
}

raw_datasets = load_dataset("json", data_files=data_files)
print(raw_datasets)


DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 116488
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 6130
    })
})


In [6]:
with open(os.path.join(DATA_DIR, "label_map.json"), 'r', encoding='utf-8') as f:
    ID_TO_LABEL_RAW = json.load(f)
    id2label = {int(k): v for k, v in ID_TO_LABEL_RAW.items()}
    label2id = {v: k for k, v in id2label.items()}

LABELS = list(id2label.values())
NUM_LABELS = len(LABELS)

print(f"Loaded {NUM_LABELS} labels.")
# print(LABELS)

Loaded 28 labels.


In [7]:
def align_labels_with_tokens(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LEN
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) # Special tokens
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx]) # First subword gets label
            else:
                label_ids.append(-100) # Subsequent subwords ignored
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


In [8]:
"""
looks like this:
{
  "tokens": ["Hello", ",", "world", "!"],
  "ner_tags": [0, 1, 0, 2]
}
"""

'\nlooks like this:\n{\n  "tokens": ["Hello", ",", "world", "!"],\n  "ner_tags": [0, 1, 0, 2]\n}\n'

# Model preparation

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
seqeval = evaluate.load("seqeval")

from custom_trainer import FocalLossTrainer, CRFTrainer

data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
class BertCRFForTokenClassification(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.num_labels = num_labels
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
        self.crf = CRF(num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, labels=None):
        outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        sequence_output = self.dropout(outputs.last_hidden_state)
        emissions = self.classifier(sequence_output)

        if labels is not None:
            mask = attention_mask.type(torch.uint8)
            safe_labels = torch.where(labels == -100, torch.zeros_like(labels), labels)
            loss = -self.crf(emissions, safe_labels, mask=mask, reduction='mean')
            return (loss, emissions)
        else:
            mask = attention_mask.type(torch.uint8) if attention_mask is not None else None
            predictions = self.crf.decode(emissions, mask=mask)
            return emissions

In [ ]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        LABELS[p]
        for prediction, label in zip(predictions, labels)
        for p, l in zip(prediction, label)
        if l != -100
    ]

    true_labels = [
        LABELS[l]
        for prediction, label in zip(predictions, labels)
        for p, l in zip(prediction, label)
        if l != -100
    ]

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, true_predictions, average='weighted', zero_division=0
    )
    accuracy = accuracy_score(true_labels, true_predictions)

    report = classification_report(true_labels, true_predictions, zero_division=0)
    print("\n" + report)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy,
    }

# Training Pipeline

In [ ]:
#  TRAINING FLAGS # 
TRAIN_BASELINE = True
USE_CRF        = True
USE_KFOLD      = True
# # # # # # # # # # 

if DEBUG:
    NUM_EPOCHS    = 1
    FT_EPOCHS     = 1
    logging_steps = 5
else:
    NUM_EPOCHS    = 3
    FT_EPOCHS     = 5
    logging_steps = 100

suffix = "crf" if USE_CRF else "focal"
BASELINE_MODEL_DIR  = os.path.join(MODEL_DIR, f"baseline_model_{suffix}")
FINETUNED_MODEL_DIR = os.path.join(MODEL_DIR, f"finetuned_lorugec_{suffix}")

print(f"TRAIN_BASELINE={TRAIN_BASELINE}  USE_CRF={USE_CRF}  USE_KFOLD={USE_KFOLD}")
print(f"Baseline  → {BASELINE_MODEL_DIR}")
print(f"Finetuned → {FINETUNED_MODEL_DIR}")

In [ ]:
def get_model(use_crf=USE_CRF, load_from_path=None):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    if use_crf:
        model = BertCRFForTokenClassification(MODEL_NAME, num_labels=NUM_LABELS)
        if load_from_path is not None:
            weights_path = os.path.join(load_from_path, "pytorch_model.bin")
            model.load_state_dict(
                torch.load(weights_path, map_location=device, weights_only=True)
            )
            print(f"[CRF] Loaded weights from {weights_path}")
        return model.to(device)

    if load_from_path is not None:
        return AutoModelForTokenClassification.from_pretrained(
            load_from_path,
            num_labels=NUM_LABELS,
            id2label=id2label,
            label2id=label2id,
        ).to(device)

    return AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=id2label,
        label2id=label2id,
    ).to(device)

In [ ]:
def get_trainer(model_to_train, train_ds, val_ds,
                output_dir="training-output", learning_rate=2e-5,
                num_epochs=3, eval_strategy="epoch", save_strategy="epoch"):
    args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy=eval_strategy,
        save_strategy=save_strategy,
        learning_rate=learning_rate,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=num_epochs,
        weight_decay=0.01,
        logging_steps=logging_steps,
        gradient_accumulation_steps=GRAD_ACCUM,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

    TrainerClass = CRFTrainer if USE_CRF else FocalLossTrainer

    return TrainerClass(
        model=model_to_train,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )


def save_model(model_to_save, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    if USE_CRF:
        torch.save(
            model_to_save.state_dict(),
            os.path.join(save_dir, "pytorch_model.bin"),
        )
    else:
        model_to_save.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    print(f"Model saved → {save_dir}")

## Baseline Training

In [ ]:
tokenized_baseline = raw_datasets.map(align_labels_with_tokens, batched=True)

if DEBUG:
    tokenized_baseline["train"] = tokenized_baseline["train"].select(range(50))
    tokenized_baseline["validation"] = tokenized_baseline["validation"].select(range(10))

if TRAIN_BASELINE:
    print(f"Training baseline ({'CRF' if USE_CRF else 'Focal Loss'})…")

    model_baseline = get_model(use_crf=USE_CRF)

    trainer_baseline = get_trainer(
        model_baseline,
        tokenized_baseline["train"],
        tokenized_baseline["validation"],
        output_dir=f"bert-{suffix}-baseline",
        learning_rate=2e-5,
        num_epochs=NUM_EPOCHS,
    )
    trainer_baseline.train()

    save_model(model_baseline, BASELINE_MODEL_DIR)

    del model_baseline, trainer_baseline
    torch.cuda.empty_cache()
else:
    print(f"Skipping baseline (TRAIN_BASELINE=False).\n"
          f"Expecting saved model at: {BASELINE_MODEL_DIR}")

## LORuGEC Fine-Tuning

In [ ]:
lorugec_files = {
    "train_all":  os.path.join(DATA_DIR, "trainall_lorugec.json"),
    "train":      os.path.join(DATA_DIR, "train_lorugec.json"),
    "validation": os.path.join(DATA_DIR, "val_lorugec.json"),
    "test":       os.path.join(DATA_DIR, "bench_test_lorugec.json"),
}

raw_lorugec = load_dataset("json", data_files=lorugec_files)
tokenized_lorugec = raw_lorugec.map(align_labels_with_tokens, batched=True)
print(raw_lorugec)

In [ ]:
FT_LR   = 1e-5
N_FOLDS = 5

if USE_KFOLD:
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    full_ds = tokenized_lorugec["train_all"]
    fold_results = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(range(len(full_ds)))):
        print(f"\n{'='*50}")
        print(f"  Fold {fold + 1}/{N_FOLDS}")
        print(f"{'='*50}")

        model_fold = get_model(use_crf=USE_CRF, load_from_path=BASELINE_MODEL_DIR)

        trainer_fold = get_trainer(
            model_fold,
            full_ds.select(train_idx),
            full_ds.select(val_idx),
            output_dir=f"bert-lorugec-fold{fold}",
            learning_rate=FT_LR,
            num_epochs=FT_EPOCHS,
            save_strategy="no",
        )
        trainer_fold.train()

        metrics = trainer_fold.evaluate()
        print(f"  → F1={metrics['eval_f1']:.4f}  "
              f"P={metrics['eval_precision']:.4f}  "
              f"R={metrics['eval_recall']:.4f}")
        fold_results.append(metrics)

        del model_fold, trainer_fold
        torch.cuda.empty_cache()

    print(f"\n{'='*50}")
    print("  KFold Summary")
    print(f"{'='*50}")
    for key in ("eval_f1", "eval_precision", "eval_recall", "eval_accuracy"):
        vals = [r[key] for r in fold_results]
        print(f"  {key:20s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")
    print(f"{'='*50}")

    # Final model on train_all
    print("\nTraining final production model on 100% of train_all…")
    model_final = get_model(use_crf=USE_CRF, load_from_path=BASELINE_MODEL_DIR)

    trainer_final = get_trainer(
        model_final, full_ds, full_ds,
        output_dir="bert-lorugec-final",
        learning_rate=FT_LR,
        num_epochs=FT_EPOCHS,
        eval_strategy="no",
        save_strategy="no",
    )
    trainer_final.train()
    save_model(model_final, FINETUNED_MODEL_DIR)

    del model_final, trainer_final
    torch.cuda.empty_cache()

else:
    # Simple train / val split
    print("Simple fine-tuning (train / val split)…")
    model_ft = get_model(use_crf=USE_CRF, load_from_path=BASELINE_MODEL_DIR)

    trainer_ft = get_trainer(
        model_ft,
        tokenized_lorugec["train"],
        tokenized_lorugec["validation"],
        output_dir="bert-lorugec-simple",
        learning_rate=FT_LR,
        num_epochs=FT_EPOCHS,
    )
    trainer_ft.train()
    save_model(model_ft, FINETUNED_MODEL_DIR)

    del model_ft, trainer_ft
    torch.cuda.empty_cache()

## LORuGEC Benchmark

In [ ]:
print(f"Loading final model from {FINETUNED_MODEL_DIR}…")
model_bench = get_model(use_crf=USE_CRF, load_from_path=FINETUNED_MODEL_DIR)

trainer_bench = get_trainer(
    model_bench,
    tokenized_lorugec["test"],
    tokenized_lorugec["test"],
    output_dir="benchmark-eval",
    eval_strategy="no",
    save_strategy="no",
)

bench = trainer_bench.evaluate(tokenized_lorugec["test"])

print(f"\n{'='*50}")
print("  OFFICIAL BENCHMARK  (bench_test_lorugec.json)")
print(f"{'='*50}")
print(f"  Precision : {bench['eval_precision']:.4f}")
print(f"  Recall    : {bench['eval_recall']:.4f}")
print(f"  F1        : {bench['eval_f1']:.4f}")
print(f"  Accuracy  : {bench['eval_accuracy']:.4f}")
print(f"{'='*50}")

# Restoration Demo

In [ ]:
import re

model = get_model(use_crf=USE_CRF, load_from_path=FINETUNED_MODEL_DIR)
model.eval()


def reconstruct_text(tokens, labels, id2label):
    text = ""
    for token, label_id in zip(tokens, labels):
        if token in ["[CLS]", "[SEP]", "[PAD]"]:
            continue
        punct = id2label[label_id]
        if punct == "O":
            text += token + " "
        else:
            text += token + punct + " "
    return text.strip()


def restore_punctuation(text):
    device = next(model.parameters()).device

    words = [m.group() for m in re.finditer(r"[\w]+(?:-[\w]+)*", text)]
    if not words:
        return ""

    inputs = tokenizer(
        words,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN,
        is_split_into_words=True,
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

        if hasattr(model, "crf"):
            emissions = outputs[1] if isinstance(outputs, tuple) else outputs
            mask = inputs["attention_mask"].byte()
            predictions = model.crf.decode(emissions, mask=mask)[0]
        else:
            logits = outputs.logits if hasattr(outputs, "logits") else outputs
            predictions = torch.argmax(logits, dim=2)[0].cpu().tolist()

    word_ids = inputs.word_ids()
    restored_text = ""
    previous_word_idx = None

    for i, word_idx in enumerate(word_ids):
        if word_idx is None or word_idx == previous_word_idx:
            continue
        original_word = words[word_idx]
        pred_id = predictions[i] if i < len(predictions) else 0
        punct = id2label.get(pred_id, "O")

        if punct == "O":
            restored_text += original_word + " "
        else:
            restored_text += original_word + punct + " "
        previous_word_idx = word_idx

    return re.sub(r'\s+', ' ', restored_text).strip()

In [ ]:
tests = [
    "Мама мыла раму а папа читал газету",
    "Однако мы решили не идти в кино потому что пошел дождь",
    "Он сказал Привет как дела",
    "Я говорю ей Что думаешь А она мне Да ничего отстань уже",
    "После многих страданий А А Пушкин всё-таки написал свои стишки",
]

for t in tests:
    print(f"Input:    {t}")
    print(f"Restored: {restore_punctuation(t)}\n")